# RAG CoT
(Chain of Thought)

RAG 파이프라인에서 LLM이 단순한 정보 조합을 넘어서 단계적 사고를 통해 논리적 답변을 할 수 있도록 한다.

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://apac.api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

In [3]:
# 가상 검색기
from langchain_core.documents import Document

def retrieve_vectordb(query=None):
    return [
        Document(page_content='대한민국의 수도는 서울입니다. 서울은 한강을 끼고 발달한 도시입니다.'),
        Document(page_content='서울의 대표적인 관광지는 경복궁, 남산타워, 명동 등이 있습니다.'),
        Document(page_content='서울의 인구는 약 천만 명이고, 교통 문화 인프라가 잘 갖추어져 있습니다.')
    ]

retrieve_vectordb()

[Document(metadata={}, page_content='대한민국의 수도는 서울입니다. 서울은 한강을 끼고 발달한 도시입니다.'),
 Document(metadata={}, page_content='서울의 대표적인 관광지는 경복궁, 남산타워, 명동 등이 있습니다.'),
 Document(metadata={}, page_content='서울의 인구는 약 천만 명이고, 교통 문화 인프라가 잘 갖추어져 있습니다.')]

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

llm = init_chat_model('gpt-5.6-luna')
prompt = ChatPromptTemplate.from_template('''
당신은 데이터를 분석해서 논리적인 결론을 도출하는 전문가 챗봇입니다.
아래 [검색된 문서]를 바탕으로 사용자의 [질문]에 대해 답변하세요.

[검색된 문서]
{context}

[질문]
{question}

[지시사항]
다음의 단계에 따라 사고한 후, 답변을 작성하세요.
1. **핵심데이터 정리**: 문서에서 사용자 질문과 관련한 팩트를 추출해보세요.
2. **상호관계 분석**: 각 항목별로 어떤 상관관계/시너지를 도출하는지 고민하세요.
3. **논리적 서술**: 위의 사고한 내용을 토대로 사용자 질문에 대한 답변을 준비하세요.
4. **최종 답변**: 서론-본론-결론 구조에 맞춰 완성된 답변을 작성하세요.
''')
output_parser = StrOutputParser()

chain = prompt | llm | output_parser

question = '서울의 인구, 관광지, 교통인프라를 종합해서 여행하기 좋은 이유를 논리적으로 설명해주세요.'
retrieved_docs = retrieve_vectordb(question)
# 문서 본문만 뽑아서 하나의 문자열 context로 생성
context = '\n\n'.join([doc.page_content for doc in retrieved_docs])

response = chain.invoke({'context': context, 'question': question})
print(response)

### 서론  
서울은 약 천만 명이 거주하는 대도시로, 다양한 관광지와 잘 갖추어진 교통·문화·인프라를 갖추고 있어 여행하기 좋은 도시입니다.

### 본론  
먼저 서울에는 경복궁, 남산타워, 명동 등 대표적인 관광지가 있습니다. 경복궁에서는 전통적인 역사와 문화를, 남산타워에서는 서울의 도시 경관을, 명동에서는 도심의 활기와 상업 문화를 경험할 수 있습니다. 이처럼 서로 다른 성격의 관광지가 있어 여행객은 한 도시 안에서도 다양한 여행 목적을 충족할 수 있습니다.

또한 서울은 약 천만 명이 거주하는 대도시이기 때문에 관광, 쇼핑, 음식, 문화 등 다양한 시설과 활동이 발달해 있습니다. 여기에 교통과 문화 인프라가 잘 갖추어져 있어 여러 관광지를 이동하고 이용하기에 편리합니다. 즉, 관광지가 다양할 뿐 아니라 이를 연결하고 뒷받침하는 교통·문화 기반도 마련되어 있어 여행의 효율성과 편의성이 높습니다.

더불어 서울은 한강을 끼고 발달한 도시라는 점도 장점입니다. 역사적 명소와 현대적인 도심뿐 아니라 한강을 중심으로 한 도시 경관까지 함께 즐길 수 있어 여행 경험이 더욱 풍부해집니다.

### 결론  
종합하면 서울은 약 천만 명 규모의 대도시가 제공하는 다양한 문화·상업적 볼거리, 경복궁·남산타워·명동 같은 대표 관광지, 그리고 편리한 교통·문화 인프라가 서로 시너지를 내는 곳입니다. 따라서 짧은 시간에도 여러 유형의 관광을 편리하게 즐길 수 있어 여행하기 좋은 도시라고 할 수 있습니다.
